This code tries to improve the preivous results obtained with CLIP in zero shot mode. To do so we could exploit the support images, and build a little classifier on top of the Image Encoder that that can be trained on the support images. 

In [1]:
# Paths definition

import os

DATASET_PATH = '/kaggle/input/competitions/aml-group-project/release'
TRAIN_CSV_PATH = '/kaggle/input/competitions/aml-group-project/release/train.csv'
IMGS_PATH = '/kaggle/input/competitions/aml-group-project/release/images'
TEST_CSV_PATH = '/kaggle/input/competitions/aml-group-project/release/test_episodes_release.csv'
SUBMISSION_CSV_PATH = '/kaggle/input/competitions/aml-group-project/release/sample_submission.csv'

In [2]:
import pandas as pd
import numpy as np

def generate_episode_from_train(train_df, way=5, shot=5, query_per_class=5):
    
    """
    Generate a fictitious episode starting from the train dataframe.
    Returns back the support and query subsets with visible labels.
    This functions is usefull to have some local tests before submitting on kaggle.
    """
    
    # 1. Extract the available classes of the train set and choose 5 random classes
    available_classes = train_df['label'].unique()
    selected_classes = np.random.choice(available_classes, size=way, replace=False)
    
    support_rows = []
    query_rows   = []
    
    # 2. For each class in selected_classes, we extract the support and query images
    for idx, cls in enumerate(selected_classes):

        # 3. Extract only the rows of the df corresponding to the class selected in the for loop
        class_subset = train_df[train_df['label'] == cls]
        
        # 4. Extract 10 random rows of this subset of the original df (5 for the support and 5 for the query)
        sampled_rows = class_subset.sample(n= shot + query_per_class, replace=False)
        
        # 5. Select 5 rows for the support and 5 for the query
        support_part = sampled_rows.iloc[:shot].copy()
        query_part = sampled_rows.iloc[shot:].copy()
        
        # 6. Assign labels to each class (from 0 to 4)
        support_part['local_label'] = idx
        query_part['local_label'] = idx
        
        support_rows.append(support_part)
        query_rows.append(query_part)
        
    # Merge everything to get the final structure for the hyphotetical episode
    episode_support_df = pd.concat(support_rows).sample(frac=1).reset_index(drop=True)
    episode_query_df = pd.concat(query_rows).sample(frac=1).reset_index(drop=True)
    
    return episode_support_df, episode_query_df

In [ ]:
 pip install ftfy regex tqdm git+https://github.com/openai/CLIP.git

## Implementation of the Linear Probe

In [5]:
import random
import numpy as np
import torch

def set_reproducibility(seed=42):
    """
    Fissa tutti i seed casuali per garantire che l'esecuzione del notebook
    produca esattamente gli stessi identici risultati a ogni run.
    """
    # 1. Python standard
    random.seed(seed)
    
    # 2. NumPy
    np.random.seed(seed)
    
    # 3. PyTorch (CPU)
    torch.manual_seed(seed)
    
    # 4. PyTorch (GPU / CUDA)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # Per configurazioni multi-GPU (se presenti)
        
        # 5. Forziamo gli algoritmi deterministici di CUDA 
        # (Disattiva i benchmark dinamici che potrebbero introdurre variabilità float)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    print(f"[INFO] Seed globale impostato su: {seed}. Risultati ora riproducibili al 100%.")

# Eseguiamo la funzione subito
set_reproducibility(seed=42)

[INFO] Seed globale impostato su: 42. Risultati ora riproducibili al 100%.


In [9]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
import os
from tqdm import tqdm
import clip

class FeatureLinearProbe(nn.Module):
    def __init__(self, feature_dim, num_classes):
        super().__init__()
        # Nota: ViT-L/14 sputa fuori 768 dimensioni, non più 512!
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, image_features):
        return self.classifier(image_features)

# PARAMETRI
WAY = 5
SHOT = 5
QUERY_PER_CLASS = 5
NUM_EPISODES = 100

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used: {device}")

# MIGLIORIA 1: Carichiamo il modello LARGE (ViT-L/14)
print("[INFO] Loading CLIP ViT-L/14 (Large Model)...")
model, preprocess = clip.load("ViT-L/14", device=device)
model.eval()

df_train = pd.read_csv(TRAIN_CSV_PATH)
all_upgraded_accuracies = []

print(f"Inizio valutazione LINEAR PROBE POTENZIATO su {NUM_EPISODES} episodi...")

for ep in tqdm(range(NUM_EPISODES)):
    support_df, query_df = generate_episode_from_train(df_train, way=WAY, shot=SHOT, query_per_class=QUERY_PER_CLASS)

    # --- ESTRAZIONE SUPPORT ---
    support_embeddings = []
    support_labels = torch.tensor(support_df['local_label'].values, dtype=torch.long).to(device)

    for _, row in support_df.iterrows():
        img_path = os.path.join(IMGS_PATH, row['filename'])
        img = Image.open(img_path).convert('RGB')
        img_input = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            img_feature = model.encode_image(img_input)
            support_embeddings.append(img_feature)
            
    support_embeddings = torch.cat(support_embeddings, dim=0).float() 

    # --- ESTRAZIONE QUERY ---
    query_embeddings = []
    ground_truth_labels = query_df['local_label'].values

    for _, row in query_df.iterrows():
        img_path = os.path.join(IMGS_PATH, row['filename'])
        img = Image.open(img_path).convert('RGB')
        img_input = preprocess(img).unsqueeze(0).to(device)
        with torch.no_grad():
            img_feature = model.encode_image(img_input)
            query_embeddings.append(img_feature)

    query_embeddings = torch.cat(query_embeddings, dim=0).float() 

    # MIGLIORIA 2: Feature Centering & Normalization (Raddrizza lo spazio geometrico)
    # Calcoliamo la media combinata di support e query per questo specifico episodio
    combined_features = torch.cat([support_embeddings, query_embeddings], dim=0)
    episode_mean = combined_features.mean(dim=0, keepdim=True)
    
    # Sottraiamo la media e applichiamo la normalizzazione L2 pulita
    support_embeddings = support_embeddings - episode_mean
    support_embeddings = support_embeddings / support_embeddings.norm(dim=-1, keepdim=True)
    
    query_embeddings = query_embeddings - episode_mean
    query_embeddings = query_embeddings / query_embeddings.norm(dim=-1, keepdim=True)

    # --- ADDESTRAMENTO LINEAR PROBE ---
    # Cambiamo la dimensione a 768 perché stiamo usando il modello Large
    probe = FeatureLinearProbe(feature_dim=768, num_classes=WAY).to(device)
    
    # Aumentiamo leggermente il weight_decay (regolarizzazione L2) per simulare l'effetto "più dati" ed evitare overfitting
    optimizer = torch.optim.AdamW(probe.parameters(), lr=5e-3, weight_decay=1e-1)
    loss_fn = nn.CrossEntropyLoss()

    probe.train()
    for epoch in range(20): # Alziamo a 20 epoche visto che lo spazio è più grande
        logits = probe(support_embeddings)
        loss = loss_fn(logits, support_labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # --- INFERENZA ---
    probe.eval()
    with torch.no_grad():
        query_logits = probe(query_embeddings)
        predictions = query_logits.argmax(dim=1).cpu().numpy()

    ep_accuracy = np.mean(predictions == ground_truth_labels)
    all_upgraded_accuracies.append(ep_accuracy)

# STATISTICHE
mean_accuracy = np.mean(all_upgraded_accuracies)
std_accuracy = np.std(all_upgraded_accuracies)
confidence_interval = 1.96 * (std_accuracy / np.sqrt(NUM_EPISODES))

print("\n==================================================")
print("   RISULTATI FINALI (LINEAR PROBE UPGRADED V2)    ")
print("==================================================")
print(f"Average accuracy on {NUM_EPISODES} episodes: {mean_accuracy * 100:.2f}%")
print(f"Confidence Interval (95%):    ±{confidence_interval * 100:.2f}%")
print(f"Format for table:              {mean_accuracy * 100:.2f}% ± {confidence_interval * 100:.2f}%")
print("==================================================")

Device used: cuda
[INFO] Loading CLIP ViT-L/14 (Large Model)...
Inizio valutazione LINEAR PROBE POTENZIATO su 100 episodi...


100%|██████████| 100/100 [01:35<00:00,  1.04it/s]


   RISULTATI FINALI (LINEAR PROBE UPGRADED V2)    
Average accuracy on 100 episodes: 90.36%
Confidence Interval (95%):    ±1.49%
Format for table:              90.36% ± 1.49%


- Accuracy Modello B-16: 90.56%
- Accuracy Modello L-14: 88.89%
- Accuracy Modello L-14 con feature centering e normalization + weight decay e lr + bassi: 90.36%
After these attemtps I had the feeling that not using the text encoder would have a huge impact. What if there was a way to extract textual templates from the images and use them to feed the text encoder? then I could use both the text encoder and the image encoder. Idea: use a model to generate captions for the images and then use standard CLIp